# GDN metrics notebook

This notebook processes `T_Network` and `N_Network`, validates the required inputs, exports results to `result_files/`, and skips incomplete networks without breaking execution.

## Import libraries
Load the helper functions and notebook dependencies.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

import gdn_analysis
from gdn_analysis import (
    NETWORK_ORDER,
    compute_graph_metrics,
    compute_sample_activation_metrics,
    compute_vertex_metrics,
    discover_available_networks,
    export_figure,
    export_graph_summary,
    export_isolated_nodes,
    export_sample_metrics,
    export_vertex_metrics,
    plot_degree_vs_frequency,
    plot_spontaneous_vs_total,
    summarize_run,
)

## Discover available networks
Detect which network folders contain the required input files.

In [2]:
ROOT = Path(gdn_analysis.__file__).resolve().parent
bundles, skipped_networks = discover_available_networks(ROOT)
run_status = summarize_run(skipped_networks, list(bundles.keys()))
display(run_status)

,network,status,details
0,T_Network,processed,all required inputs found
1,N_Network,processed,all required inputs found


## Compute vertex-level metrics
Calculate node name, frequency, in-degree, and out-degree for each available network.

In [3]:
vertex_metrics_by_network = {}
for network_name in NETWORK_ORDER:
    if network_name not in bundles:
        continue
    bundle = bundles[network_name]
    vertex_metrics = compute_vertex_metrics(bundle)
    vertex_metrics_by_network[network_name] = vertex_metrics
    export_vertex_metrics(vertex_metrics, bundle['results_dir'])
    print(f"[{network_name}] vertex_metrics.csv exported to {bundle['results_dir']}")
    display(vertex_metrics.head())

[T_Network] vertex_metrics.csv exported to C:\Frank\Work\ICIMAF\GDN\T_Network\result_files


,row_index,node_id,representative_original_id,name,frequency,in_degree,out_degree
0,0,0,0,ENSG00000146083,0.254509,21,3
1,1,1,1,ENSG00000198242,0.222445,28,6
2,2,2,2,ENSG00000167700,0.294589,23,8
3,3,3,3,ENSG00000153561,0.110220,2,35
4,4,4,4,ENSG00000275479,0.104208,0,221


[N_Network] vertex_metrics.csv exported to C:\Frank\Work\ICIMAF\GDN\N_Network\result_files


,row_index,node_id,representative_original_id,name,frequency,in_degree,out_degree
0,0,0,0,ENSG00000111863,0.865385,31,13
1,1,1,2,ENSG00000164116,0.942308,17,0
2,2,2,3,ENSG00000176928,0.826923,8,22
3,3,3,4,ENSG00000155974,0.942308,24,0
4,4,4,5,ENSG00000279952,0.884615,30,40


## Compute graph-level metrics
Summarize degree statistics, isolated nodes, density, and distance-based metrics.

In [4]:
graph_metrics_by_network = {}
for network_name in NETWORK_ORDER:
    if network_name not in bundles:
        continue
    bundle = bundles[network_name]
    vertex_metrics = vertex_metrics_by_network[network_name]
    graph_metrics = compute_graph_metrics(vertex_metrics, bundle)
    graph_metrics_by_network[network_name] = graph_metrics
    export_isolated_nodes(vertex_metrics, bundle['results_dir'])
    export_graph_summary(graph_metrics, bundle['results_dir'])
    display(pd.DataFrame([graph_metrics]))

,network,label,n_nodes,n_edges,n_samples,density,max_in_degree,mean_in_degree,max_out_degree,mean_out_degree,isolated_nodes,orphan_nodes,childless_nodes,maximum_diameter,mean_diameter,directed_reachable_diameter,directed_mean_reachable_distance,largest_wcc_size,largest_wcc_undirected_diameter,distance_method
0,T_Network,T,6138,102362,499,0.002717,237,16.676768,221,16.676768,151,1624,582,14,2.565239,14,2.565239,5967,9,scipy_shortest_path


,network,label,n_nodes,n_edges,n_samples,density,max_in_degree,mean_in_degree,max_out_degree,mean_out_degree,isolated_nodes,orphan_nodes,childless_nodes,maximum_diameter,mean_diameter,directed_reachable_diameter,directed_mean_reachable_distance,largest_wcc_size,largest_wcc_undirected_diameter,distance_method
0,N_Network,N,499,4984,52,0.020056,60,9.987976,165,9.987976,1,9,92,10,2.463631,10,2.463631,498,7,scipy_shortest_path


## Generate degree-frequency plots
Export the two-panel plot relating frequency to in-degree and out-degree.

In [5]:
for network_name in NETWORK_ORDER:
    if network_name not in bundles:
        continue
    bundle = bundles[network_name]
    vertex_metrics = vertex_metrics_by_network[network_name]
    figure1 = plot_degree_vs_frequency(vertex_metrics, bundle)
    export_figure(figure1, bundle['results_dir'], 'figure1_degree_vs_frequency')
    print(f"[{network_name}] Figure 1 exported")

[T_Network] Figure 1 exported
[N_Network] Figure 1 exported


## Compute sample activation metrics
Measure total active genes and spontaneously active genes for each sample.

In [6]:
sample_metrics_by_network = {}
for network_name in NETWORK_ORDER:
    if network_name not in bundles:
        continue
    bundle = bundles[network_name]
    sample_metrics = compute_sample_activation_metrics(bundle)
    sample_metrics_by_network[network_name] = sample_metrics
    export_sample_metrics(sample_metrics, bundle['results_dir'], network_name)
    display(sample_metrics.head())

,sample_index,spontaneous_active,total_active
0,0,669,2002
1,1,389,909
2,2,553,1490
3,3,161,270
4,4,338,943


,sample_index,spontaneous_active,total_active
0,0,10,496
1,1,9,488
2,2,13,466
3,3,9,491
4,4,14,315


## Generate spontaneous activation plot
Export the sample-level comparison between spontaneous and total activations.

In [7]:
for network_name in NETWORK_ORDER:
    if network_name not in bundles:
        continue
    bundle = bundles[network_name]
    sample_metrics = sample_metrics_by_network[network_name]
    figure2 = plot_spontaneous_vs_total(sample_metrics, bundle)
    export_figure(figure2, bundle['results_dir'], 'figure2_spontaneous_vs_total')
    print(f"[{network_name}] Figure 2 exported")

[T_Network] Figure 2 exported
[N_Network] Figure 2 exported
